# 01 - Exploración de los datos

Revisa el resultado del preprocesamiento (`manifest.csv`): cuántas personas, videos y
frames hay por fuente, split y tipo de ataque; el balance de clases; los frames
descartados por no tener rostro; y ejemplos de los recortes.

Requiere haber ejecutado antes `extraer_frames`, `recortar_rostro` y `dividir_por_persona` (ver README).

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.data.imagenes import leer_imagen

COLOR_REAL, COLOR_ATAQUE, COLOR_TEXTO = "#2a78d6", "#eb6834", "#52514e"

manifest = pd.read_csv(config.MANIFEST, keep_default_na=False)
print(f"{len(manifest)} frames con rostro")
manifest.head()

## 1. Personas, videos y frames por fuente y split

In [ ]:
conteos = manifest.groupby(["fuente", "split"]).agg(
    personas=("id_persona", "nunique"),
    videos=("id_video", "nunique"),
    frames=("ruta_frame", "size"),
).reindex(config.SPLITS, level="split")
conteos

> En LCC FASD el split de test (evaluation oficial) no trae identificadores de persona ni de
> video: aparece como una sola "persona" y cada imagen cuenta como un video.

## 2. Frames y videos por tipo de ataque

In [ ]:
por_tipo = manifest.pivot_table(index=["fuente", "split"], columns="tipo_ataque",
                                values="ruta_frame", aggfunc="size", fill_value=0).reindex(config.SPLITS, level="split")
videos_por_tipo = manifest.pivot_table(index=["fuente", "split"], columns="tipo_ataque",
                                       values="id_video", aggfunc="nunique", fill_value=0).reindex(config.SPLITS, level="split")
print("Frames:")
display(por_tipo)
print("Videos:")
display(videos_por_tipo)

## 3. Balance de clases

In [ ]:
balance = manifest.groupby(["fuente", "split"])["etiqueta"].agg(
    reales=lambda e: int((e == config.ETIQUETA_REAL).sum()),
    ataques=lambda e: int((e == config.ETIQUETA_ATAQUE).sum()),
).reindex(config.SPLITS, level="split")
balance["% ataque"] = (100 * balance["ataques"] / (balance["reales"] + balance["ataques"])).round(1)
display(balance)

etiquetas = [f"{f} / {s}" for f, s in balance.index]
x = np.arange(len(balance))
fig, ax = plt.subplots(figsize=(1.6 * len(balance) + 2, 4))
ax.bar(x - 0.2, balance["reales"], width=0.38, color=COLOR_REAL, label="Real")
ax.bar(x + 0.2, balance["ataques"], width=0.38, color=COLOR_ATAQUE, label="Ataque")
ax.set_xticks(x, etiquetas)
ax.set_ylabel("Frames", color=COLOR_TEXTO)
ax.set_title("Balance de clases por fuente y split")
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)
ax.grid(axis="y", color="#e6e5e0")
ax.set_axisbelow(True)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

> LCC FASD está muy desbalanceado (aprox. 85-96 % ataques). Por eso la exactitud sola no es
> una buena métrica y se reportan APCER, BPCER y ACER. Al entrenar conviene compensar el
> desbalance, por ejemplo dando más peso a los frames reales en la binary cross-entropy.

## 4. Frames descartados (sin rostro detectado)

In [ ]:
frames = pd.read_csv(config.MANIFEST_FRAMES, keep_default_na=False)
rostros = pd.read_csv(config.MANIFEST_ROSTROS, keep_default_na=False)
descartes = pd.DataFrame({
    "frames": frames.groupby(["fuente", "tipo_ataque"]).size(),
    "con_rostro": rostros.groupby(["fuente", "tipo_ataque"]).size(),
}).fillna(0).astype(int)
descartes["descartados"] = descartes["frames"] - descartes["con_rostro"]
descartes["% descartado"] = (100 * descartes["descartados"] / descartes["frames"]).round(1)
descartes

## 5. Ejemplos de recortes

In [ ]:
def mostrar_ejemplos(manifest, n_por_fila=6, semilla=config.SEMILLA):
    grupos = list(manifest.groupby(["fuente", "tipo_ataque"]))
    fig, ejes = plt.subplots(len(grupos), n_por_fila, figsize=(1.8 * n_por_fila, 2 * len(grupos)),
                             squeeze=False)
    for fila, ((fuente, tipo), grupo) in enumerate(grupos):
        un_frame = grupo.groupby("id_video").head(1)
        muestra = un_frame.sample(min(n_por_fila, len(un_frame)), random_state=semilla)
        for col in range(n_por_fila):
            ax = ejes[fila, col]
            ax.axis("off")
            if col < len(muestra):
                imagen = leer_imagen(config.ruta_absoluta(muestra.iloc[col]["ruta_frame"]))
                ax.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
        ejes[fila, 0].set_title(f"{fuente} - {tipo}", loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()

mostrar_ejemplos(manifest)

## 6. Así ve el rostro la regresión logística (64×64 en gris)

In [ ]:
ejemplos = manifest.groupby("etiqueta").sample(4, random_state=config.SEMILLA)
fig, ejes = plt.subplots(2, 4, figsize=(8, 4.4))
for ax, (_, fila) in zip(ejes.ravel(), ejemplos.iterrows()):
    gris = leer_imagen(config.ruta_absoluta(fila["ruta_frame"]), gris=True)
    pequena = cv2.resize(gris, (config.TAMANO_REGRESION, config.TAMANO_REGRESION),
                         interpolation=cv2.INTER_AREA)
    ax.imshow(pequena, cmap="gray")
    ax.set_title("real" if fila["etiqueta"] == config.ETIQUETA_REAL else "ataque", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()